# 12 · Dynamic Programming

After this notebook you can tell when a problem is DP, write it top-down (memoised recursion) and bottom-up (a table), shrink the table to O(1) or O(n) memory, and solve the classics: climbing stairs, house robber, coin change, 0/1 knapsack, LIS, LCS, edit distance and grid paths.

**Time:** ~45 min · **Runs:** offline on CPU in < 1 min · **Needs:** [Recursion and backtracking](08_recursion_and_backtracking.ipynb)

## Why this matters in interviews

- DP is the category candidates fear most, yet it is a recipe: **state → recurrence → base cases → order → answer**. Saying those five things out loud before coding is what interviewers grade.
- The expected progression is brute-force recursion → memoisation → bottom-up table → space optimisation. Each step is a follow-up question.
- The same tables power real tools: edit distance drives spell-checkers and fuzzy matching; longest common subsequence is the core of `diff`.

## What interviewers ask

- "Climbing stairs" (LeetCode 70), "house robber" (198), "unique paths" (62 / 63).
- "Coin change: fewest coins" (322) and "number of ways" (518).
- "0/1 knapsack" / "partition equal subset sum" (416).
- "Longest increasing subsequence" (300) — then "in O(n log n)?"
- "Longest common subsequence" (1143), "edit distance" (72), "word break" (139).
- Follow-ups: "what is your state?", "can you do it in O(n) memory?", "why does the loop order matter?", "why is greedy wrong here?"

In [ ]:
import bisect
import itertools
import math
import random
import time
from functools import lru_cache

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e5e0", "axes.axisbelow": True, "axes.titlesize": 11,
                     "legend.frameon": False})
BLUE, ORANGE, AQUA, INK, MUTED = "#2a78d6", "#eb6834", "#1baf7a", "#0b0b0b", "#52514e"
rnd = random.Random(0)

## 1. How to spot a DP problem

**In plain English:** DP is recursion that refuses to solve the same sub-problem twice. It applies when two things are true:

1. **Overlapping subproblems** — the naive recursion asks the same question many times.
2. **Optimal substructure** — the best answer is built from best answers to smaller questions ("the best way to reach step 10 comes from the best ways to reach steps 9 and 8").

**Signals in the wording:** "number of ways", "minimum / maximum cost", "is it possible", "longest / shortest subsequence", a choice at every step (take it or skip it), and input sizes in the hundreds or thousands. Compare: **greedy** ([notebook 13](13_greedy_and_intervals.ipynb)) when one local choice is provably safe; **divide and conquer** (merge sort) when the halves never overlap.

Here is the overlap, drawn: the call tree of the naive recursion `ways(n) = ways(n − 1) + ways(n − 2)` for `n = 5`, in the order Python runs it (left branch first). Blue nodes solve a sub-problem for the first time — they run down the left edge. Every orange node repeats work already done; with a cache, the first orange node of each branch becomes an instant lookup and nothing below it runs.

In [ ]:
def call_tree(n):
    return (n, [] if n <= 1 else [call_tree(n - 1), call_tree(n - 2)])

def draw_call_tree(root, title):
    pos, seen, leaf_x = {}, set(), [0]
    def layout(node, depth, key):
        n, kids = node
        if not kids:
            x = leaf_x[0]; leaf_x[0] += 1
        else:
            xs = [layout(k, depth + 1, key + (i,)) for i, k in enumerate(kids)]
            x = sum(xs) / len(xs)
        pos[key] = (x, -depth, n, [key + (i,) for i in range(len(kids))])
        return x
    layout(root, 0, ())
    fig, ax = plt.subplots(figsize=(10, 3.6))
    for key in pos:
        x, y, n, kids = pos[key]
        for kid in kids:
            ax.plot([x, pos[kid][0]], [y, pos[kid][1]], color="#b5b3ac", lw=1.3, zorder=1)
    for key in sorted(pos):                     # tuple order = execution (pre-)order of the recursion
        x, y, n, _ = pos[key]
        repeat = n in seen
        seen.add(n)
        ax.scatter([x], [y], s=430, zorder=2, color="#fde7dc" if repeat else "#e7f0ff",
                   edgecolors=ORANGE if repeat else BLUE, linewidths=1.5)
        ax.text(x, y, f"w({n})", ha="center", va="center", fontsize=8, zorder=3)
    ax.axis("off"); ax.set_title(title)
    plt.show()
    return len(pos), len(seen)

total_calls, distinct = draw_call_tree(call_tree(5), "Naive ways(5): blue = first time a sub-problem is solved, "
                                       "orange = solved again (pure waste)")
print(f"{total_calls} calls for only {distinct} distinct sub-problems (w(0) .. w(5))")
assert (total_calls, distinct) == (15, 6)

## 2. Top-down vs bottom-up — climbing stairs (LeetCode 70)

You climb `n` stairs taking 1 or 2 steps at a time. How many distinct ways? The last move was either a 1-step (from stair `n − 1`) or a 2-step (from `n − 2`), so `ways(n) = ways(n − 1) + ways(n − 2)`, with `ways(0) = ways(1) = 1`.

Four versions, from the brute force to the interview-ready one:

In [ ]:
calls = 0
def climb_naive(n):                                   # O(2^n): recomputes everything
    global calls
    calls += 1
    return 1 if n <= 1 else climb_naive(n - 1) + climb_naive(n - 2)

@lru_cache(maxsize=None)
def climb_memo(n):                                    # top-down: O(n) time, O(n) cache + O(n) stack
    return 1 if n <= 1 else climb_memo(n - 1) + climb_memo(n - 2)

def climb_table(n):                                   # bottom-up: O(n) time, O(n) table, no recursion
    dp = [1] * (n + 1)                                # dp[i] = ways to reach stair i
    for i in range(2, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]
    return dp[n]

def climb(n):                                         # bottom-up, only the last two values: O(1) space
    prev, cur = 1, 1
    for _ in range(2, n + 1):
        prev, cur = cur, prev + cur
    return cur

for n in range(0, 26):
    assert climb_naive(n) == climb_memo(n) == climb_table(n) == climb(n)
calls = 0; climb_naive(25)
print(f"ways(25) = {climb(25):,}; the naive recursion made {calls:,} calls, the table filled 26 cells")
print(f"ways(1000) has {len(str(climb(1000)))} digits and needs no recursion at all")

| | Top-down (memoised recursion) | Bottom-up (table) |
|---|---|---|
| how you write it | the natural recursion + `@lru_cache` | loops that fill `dp[...]` in dependency order |
| which states it computes | only the ones actually reached | all of them |
| recursion depth | O(n) — can hit `RecursionError` | none |
| memory optimisation | hard | easy: keep only the rows you still need |
| best for | exploring a new problem, sparse state spaces | the final interview answer, big inputs |

### The DP recipe (say these five lines out loud)

1. **State:** what does `dp[i]` (or `dp[i][j]`) mean, in words? *"dp[i] = the number of ways to reach stair i."*
2. **Recurrence:** how is it built from smaller states? Think about the **last** decision. *"Last move was 1 or 2 steps."*
3. **Base cases:** the smallest states, answered directly. *"dp[0] = dp[1] = 1."*
4. **Order:** fill states so every dependency is ready first (usually increasing i, j).
5. **Answer:** which cell? (`dp[n]`, `dp[m][n]`, or the max over all cells.) Then ask: *do I only need the last row?* → shrink memory.

```python
def solve(inputs):
    dp = [BASE] * (N + 1)              # 1. state, 3. base cases
    for i in range(1, N + 1):          # 4. order
        dp[i] = best_of(dp[i - k] + cost for each last decision k)   # 2. recurrence
    return dp[N]                       # 5. answer
```

## 3. Classic DP problems

### Problem 1 — House robber (LeetCode 198)

Houses in a row hold `nums[i]` cash; you cannot rob two adjacent houses. Maximise the loot. `[2, 7, 9, 3, 1]` → 12 (2 + 9 + 1).

**Brute force:** try every set of non-adjacent houses, O(2^n). **DP:** state `dp[i]` = best loot from the first `i` houses; last decision: skip house `i` (`dp[i − 1]`) or rob it (`dp[i − 2] + nums[i]`). Only the last two values are needed.

In [ ]:
def rob_brute(nums):
    best = 0
    for mask in range(1 << len(nums)):
        if mask & (mask >> 1):
            continue                                  # two neighbouring houses chosen
        best = max(best, sum(x for i, x in enumerate(nums) if mask >> i & 1))
    return best

def rob(nums):
    prev2, prev1 = 0, 0                               # best up to house i-2, house i-1
    for x in nums:
        prev2, prev1 = prev1, max(prev1, prev2 + x)   # skip it, or rob it
    return prev1

assert rob([1, 2, 3, 1]) == 4 and rob([2, 7, 9, 3, 1]) == 12 and rob([]) == 0 and rob([5]) == 5
for _ in range(300):
    nums = [rnd.randint(0, 50) for _ in range(rnd.randint(0, 14))]
    assert rob(nums) == rob_brute(nums)
print("rob([2, 7, 9, 3, 1]) =", rob([2, 7, 9, 3, 1]), "| matches brute force on 300 random streets")

**Complexity:** O(n) time, O(1) space.

### Problem 2 — Coin change: fewest coins (322) and number of ways (518)

**Fewest coins:** `dp[a]` = fewest coins that make amount `a`; last decision: which coin was added last → `dp[a] = 1 + min(dp[a − c])`. **Brute force** is the same recursion without memory, and it explodes. **Greedy** (largest coin first) is *wrong* for some coin systems: coins `[1, 3, 4]`, amount 6 → greedy takes 4 + 1 + 1, DP finds 3 + 3.

In [ ]:
def coin_change(coins, amount):
    INF = float("inf")
    dp = [0] + [INF] * amount                         # dp[a] = fewest coins for amount a
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a and dp[a - c] + 1 < dp[a]:
                dp[a] = dp[a - c] + 1
    return dp[amount] if dp[amount] < INF else -1

calls = 0
def coin_change_brute(coins, amount):                 # plain recursion: exponential
    global calls
    calls += 1
    if amount == 0:
        return 0
    options = [coin_change_brute(coins, amount - c) for c in coins if c <= amount]
    options = [o for o in options if o >= 0]
    return 1 + min(options) if options else -1

def greedy_coins(coins, amount):
    count = 0
    for c in sorted(coins, reverse=True):
        count += amount // c
        amount %= c
    return count if amount == 0 else -1

assert coin_change([1, 2, 5], 11) == 3 and coin_change([2], 3) == -1 and coin_change([1], 0) == 0
for amount in range(0, 16):
    assert coin_change([1, 3, 4], amount) == coin_change_brute([1, 3, 4], amount)
calls = 0; coin_change_brute([1, 2, 5], 20)
print(f"amount 20 with coins [1, 2, 5]: brute-force recursion made {calls:,} calls; the table has 21 cells")
print("coins [1, 3, 4], amount 6 -> greedy:", greedy_coins([1, 3, 4], 6), "coins | DP:", coin_change([1, 3, 4], 6), "coins")
assert greedy_coins([1, 3, 4], 6) == 3 and coin_change([1, 3, 4], 6) == 2

**Number of ways — the loop order is the whole question.** With **coins in the outer loop**, each coin is considered once, in a fixed order, so `1 + 2` and `2 + 1` are the same combination (LeetCode 518). Swap the loops (**amounts outer**) and you count ordered sequences instead (LeetCode 377, "combination sum IV").

In [ ]:
def coin_combinations(coins, amount):                 # order does not matter
    dp = [1] + [0] * amount
    for c in coins:                                   # coins OUTER
        for a in range(c, amount + 1):
            dp[a] += dp[a - c]
    return dp[amount]

def coin_sequences(coins, amount):                    # order matters
    dp = [1] + [0] * amount
    for a in range(1, amount + 1):                    # amounts OUTER
        for c in coins:
            if c <= a:
                dp[a] += dp[a - c]
    return dp[amount]

def combinations_brute(coins, amount):
    return sum(1 for r in range(amount + 1) for combo in itertools.combinations_with_replacement(coins, r)
               if sum(combo) == amount)

print("coins [1, 2, 5], amount 5 -> combinations:", coin_combinations([1, 2, 5], 5),
      "| ordered sequences:", coin_sequences([1, 2, 5], 5))
assert coin_combinations([1, 2, 5], 5) == 4 and coin_sequences([1, 2, 5], 5) == 9
for amount in range(0, 13):
    assert coin_combinations([1, 2, 5], amount) == combinations_brute([1, 2, 5], amount)

**Complexity:** O(amount × len(coins)) time, O(amount) space for both.

### Problem 3 — 0/1 knapsack

Items with a weight and a value, a bag of capacity `W`; each item is taken **at most once**. Maximise the value.

**Brute force:** all 2^n subsets. **DP:** `dp[i][w]` = best value using the first `i` items with capacity `w`; last decision: skip item `i` (`dp[i − 1][w]`) or take it (`dp[i − 1][w − wt] + val`). **1-D version:** keep one row and loop `w` **downwards**, so `dp[w − wt]` still holds the previous row. Loop upwards and an item can be taken again — that silently solves the *unbounded* knapsack.

In [ ]:
def knapsack_brute(weights, values, cap):
    best = 0
    for r in range(len(weights) + 1):
        for combo in itertools.combinations(range(len(weights)), r):
            if sum(weights[i] for i in combo) <= cap:
                best = max(best, sum(values[i] for i in combo))
    return best

def knapsack_2d(weights, values, cap):
    n = len(weights)
    dp = [[0] * (cap + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        wt, val = weights[i - 1], values[i - 1]
        for w in range(cap + 1):
            dp[i][w] = dp[i - 1][w]                                      # skip item i
            if wt <= w:
                dp[i][w] = max(dp[i][w], dp[i - 1][w - wt] + val)        # take item i
    chosen, w = [], cap                                                  # walk back to find the items
    for i in range(n, 0, -1):
        if dp[i][w] != dp[i - 1][w]:
            chosen.append(i - 1); w -= weights[i - 1]
    return dp[n][cap], sorted(chosen)

def knapsack_1d(weights, values, cap):
    dp = [0] * (cap + 1)
    for wt, val in zip(weights, values):
        for w in range(cap, wt - 1, -1):              # DOWNWARDS: each item at most once
            dp[w] = max(dp[w], dp[w - wt] + val)
    return dp[cap]

def knapsack_1d_upwards(weights, values, cap):        # the bug: items get reused
    dp = [0] * (cap + 1)
    for wt, val in zip(weights, values):
        for w in range(wt, cap + 1):
            dp[w] = max(dp[w], dp[w - wt] + val)
    return dp[cap]

best, chosen = knapsack_2d([1, 3, 4, 5], [1, 4, 5, 7], 7)
print(f"weights [1, 3, 4, 5], values [1, 4, 5, 7], capacity 7 -> best value {best}, items {chosen}")
assert (best, chosen) == (9, [1, 2])
print("weights [2, 3], values [3, 4], capacity 6 -> downwards:", knapsack_1d([2, 3], [3, 4], 6),
      "| upwards (bug):", knapsack_1d_upwards([2, 3], [3, 4], 6))
assert knapsack_1d([2, 3], [3, 4], 6) == 7 and knapsack_1d_upwards([2, 3], [3, 4], 6) == 9
for _ in range(200):
    n = rnd.randint(0, 10)
    ws, vs = [rnd.randint(1, 10) for _ in range(n)], [rnd.randint(0, 30) for _ in range(n)]
    cap = rnd.randint(0, 30)
    assert knapsack_brute(ws, vs, cap) == knapsack_2d(ws, vs, cap)[0] == knapsack_1d(ws, vs, cap)

**Complexity:** O(n · W) time, O(W) space in 1-D. That is **pseudo-polynomial**: polynomial in the *value* W, exponential in its number of bits — knapsack is NP-hard in general, so a huge W needs other ideas.

### Problem 4 — Longest increasing subsequence (LeetCode 300)

`[10, 9, 2, 5, 3, 7, 101, 18]` → 4 (`2, 3, 7, 18`). A subsequence keeps order but may skip elements.

**O(n²) DP:** `dp[i]` = length of the longest increasing subsequence **ending at** `i` = 1 + max `dp[j]` over earlier `j` with a smaller value. **O(n log n):** keep `tails[L]` = the smallest possible tail of an increasing subsequence of length `L + 1`; each new number replaces the first tail ≥ it (binary search), or extends the list.

In [ ]:
def lis_brute(nums):
    return max((r for r in range(len(nums) + 1) for c in itertools.combinations(nums, r)
                if all(a < b for a, b in zip(c, c[1:]))), default=0)

def lis_quadratic(nums):
    dp = [1] * len(nums)
    for i in range(len(nums)):
        for j in range(i):
            if nums[j] < nums[i]:
                dp[i] = max(dp[i], dp[j] + 1)
    return max(dp, default=0)

def lis_fast(nums):
    tails = []
    for x in nums:
        i = bisect.bisect_left(tails, x)              # first tail >= x
        if i == len(tails):
            tails.append(x)                           # x extends the longest subsequence
        else:
            tails[i] = x                              # x is a smaller tail for length i + 1
    return len(tails)

assert lis_fast([10, 9, 2, 5, 3, 7, 101, 18]) == 4 and lis_fast([0, 1, 0, 3, 2, 3]) == 4
assert lis_fast([7, 7, 7, 7]) == 1 and lis_fast([]) == 0
for _ in range(200):
    nums = [rnd.randint(0, 20) for _ in range(rnd.randint(0, 11))]
    assert lis_brute(nums) == lis_quadratic(nums) == lis_fast(nums)

big = [rnd.randint(0, 10**6) for _ in range(2000)]
t0 = time.perf_counter(); a = lis_quadratic(big); t_quad = time.perf_counter() - t0
t0 = time.perf_counter(); b = lis_fast(big); t_fast = time.perf_counter() - t0
print(f"n = 2000: O(n^2) DP {1000 * t_quad:.0f} ms vs O(n log n) {1000 * t_fast:.1f} ms, both give {a}")
assert a == b and t_fast < t_quad

### Problem 5 — Longest common subsequence (LeetCode 1143)

`dp[i][j]` = LCS length of the first `i` characters of `a` and the first `j` of `b`. If the last characters match, extend the diagonal: `dp[i − 1][j − 1] + 1`. Otherwise drop one of them: `max(dp[i − 1][j], dp[i][j − 1])`. Walking back from the bottom-right corner recovers the subsequence itself.

**Brute force:** check all 2^m subsequences of `a` against `b`.

In [ ]:
def lcs_table(a, b):
    dp = np.zeros((len(a) + 1, len(b) + 1), dtype=int)
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            if a[i - 1] == b[j - 1]:
                dp[i, j] = dp[i - 1, j - 1] + 1
            else:
                dp[i, j] = max(dp[i - 1, j], dp[i, j - 1])
    return dp

def lcs_traceback(a, b, dp):
    i, j, out, path = len(a), len(b), [], [(len(a), len(b))]
    while i > 0 and j > 0:
        if a[i - 1] == b[j - 1]:
            out.append(a[i - 1]); i -= 1; j -= 1
        elif dp[i - 1, j] >= dp[i, j - 1]:
            i -= 1
        else:
            j -= 1
        path.append((i, j))
    return out[::-1], path

def is_subsequence(s, t):
    it = iter(t)
    return all(ch in it for ch in s)

def lcs_brute(a, b):
    return max(r for r in range(len(a) + 1) for c in itertools.combinations(a, r) if is_subsequence(c, b))

a, b = "ABCBDAB", "BDCABA"
dp = lcs_table(a, b)
seq, path = lcs_traceback(a, b, dp)
print(f"LCS({a}, {b}) = {''.join(seq)} (length {dp[-1, -1]})")
assert dp[-1, -1] == len(seq) == 4 and is_subsequence(seq, a) and is_subsequence(seq, b)
for _ in range(200):
    x = "".join(rnd.choice("ABC") for _ in range(rnd.randint(0, 8)))
    y = "".join(rnd.choice("ABC") for _ in range(rnd.randint(0, 8)))
    assert lcs_table(x, y)[-1, -1] == lcs_brute(x, y)

In [ ]:
def draw_dp_table(dp, rows, cols, path, title, match=None):
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.imshow(dp, cmap="Blues", vmin=0, vmax=dp.max() * 1.6)
    for i in range(dp.shape[0]):
        for j in range(dp.shape[1]):
            ax.text(j, i, dp[i, j], ha="center", va="center", fontsize=9, color=INK)
    for i, j in path:
        strong = match is not None and match(i, j)
        ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, ec=ORANGE, lw=2.4 if strong else 1.2))
    ax.set_xticks(range(len(cols) + 1), ["''"] + list(cols))
    ax.set_yticks(range(len(rows) + 1), ["''"] + list(rows))
    ax.xaxis.tick_top(); ax.grid(False)
    ax.set_title(title, pad=24)
    plt.show()

on_path = set(path)
draw_dp_table(dp, a, b, path, "LCS table: orange = traceback path; thick = a matched character",
              match=lambda i, j: i > 0 and j > 0 and a[i - 1] == b[j - 1] and (i - 1, j - 1) in on_path)

Read the table: numbers only grow to the right and downwards, and the bottom-right cell is the answer. The traceback path steps diagonally where characters match (those characters form the LCS) and sideways where one character is dropped.

**Complexity:** O(m · n) time and space (O(min(m, n)) space for the length alone, keeping two rows).

### Problem 6 — Edit distance / Levenshtein (LeetCode 72)

The minimum number of single-character **inserts, deletes and replacements** to turn `a` into `b`. `dp[i][j]` = edits to turn `a[:i]` into `b[:j]`. Base cases: `dp[i][0] = i` (delete everything), `dp[0][j] = j` (insert everything). If the last characters match, `dp[i][j] = dp[i − 1][j − 1]`; otherwise `1 + min(delete, insert, replace)`.

**Brute force:** the same recursion without a table; count its calls against the table's cells.

In [ ]:
def edit_table(a, b):
    dp = np.zeros((len(a) + 1, len(b) + 1), dtype=int)
    dp[:, 0] = np.arange(len(a) + 1)
    dp[0, :] = np.arange(len(b) + 1)
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            if a[i - 1] == b[j - 1]:
                dp[i, j] = dp[i - 1, j - 1]
            else:
                dp[i, j] = 1 + min(dp[i - 1, j],       # delete a[i-1]
                                   dp[i, j - 1],       # insert b[j-1]
                                   dp[i - 1, j - 1])   # replace a[i-1] with b[j-1]
    return dp

def edit_ops(a, b, dp):
    i, j, ops, path = len(a), len(b), [], [(len(a), len(b))]
    while i > 0 or j > 0:
        if i > 0 and j > 0 and a[i - 1] == b[j - 1] and dp[i, j] == dp[i - 1, j - 1]:
            i, j = i - 1, j - 1
        elif i > 0 and j > 0 and dp[i, j] == dp[i - 1, j - 1] + 1:
            ops.append(f"replace {a[i - 1]!r} -> {b[j - 1]!r}"); i, j = i - 1, j - 1
        elif i > 0 and dp[i, j] == dp[i - 1, j] + 1:
            ops.append(f"delete {a[i - 1]!r}"); i -= 1
        else:
            ops.append(f"insert {b[j - 1]!r}"); j -= 1
        path.append((i, j))
    return ops[::-1], path

calls = 0
def edit_naive(a, b):
    global calls
    calls += 1
    if not a or not b:
        return len(a) + len(b)
    if a[-1] == b[-1]:
        return edit_naive(a[:-1], b[:-1])
    return 1 + min(edit_naive(a[:-1], b), edit_naive(a, b[:-1]), edit_naive(a[:-1], b[:-1]))

src, dst = "kitten", "sitting"
ed = edit_table(src, dst)
ops, ed_path = edit_ops(src, dst, ed)
calls = 0; naive = edit_naive(src, dst)
print(f"edit_distance({src!r}, {dst!r}) = {ed[-1, -1]}: {ops}")
print(f"naive recursion: {calls:,} calls | table: {ed.size} cells")
assert ed[-1, -1] == naive == len(ops) == 3
assert edit_table("horse", "ros")[-1, -1] == 3 and edit_table("intention", "execution")[-1, -1] == 5
assert edit_table("", "abc")[-1, -1] == 3 and edit_table("same", "same")[-1, -1] == 0

In [ ]:
draw_dp_table(ed, src, dst, ed_path, "Edit distance kitten -> sitting: bottom-right = 3; orange = one optimal alignment")

**Complexity:** O(m · n) time; O(min(m, n)) space if you keep only two rows — which is how you would run it at scale:

In [ ]:
def levenshtein(a, b):
    """Two-row edit distance; works on strings or on lists of words."""
    if len(a) < len(b):
        a, b = b, a
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, start=1):
        cur = [i]
        for j, y in enumerate(b, start=1):
            cur.append(prev[j - 1] if x == y else 1 + min(prev[j], cur[j - 1], prev[j - 1]))
        prev = cur
    return prev[-1]

for _ in range(300):
    x = "".join(rnd.choice("abc") for _ in range(rnd.randint(0, 9)))
    y = "".join(rnd.choice("abc") for _ in range(rnd.randint(0, 9)))
    assert levenshtein(x, y) == edit_table(x, y)[-1, -1] == levenshtein(y, x)
print("two-row levenshtein == full table on 300 random pairs (and it is symmetric)")

### Problem 7 — Unique paths in a grid (LeetCode 62 / 63)

A robot in the top-left of an `m × n` grid moves only right or down. How many paths reach the bottom-right? `dp[r][c] = dp[r − 1][c] + dp[r][c − 1]` (it arrived from above or from the left); with obstacles, `dp = 0` on a blocked cell. Without obstacles there is a closed form, C(m + n − 2, m − 1) — a good cross-check. **Brute force:** walk every path with DFS.

In [ ]:
def unique_paths(grid):
    """grid of 0 (free) / 1 (blocked); returns the number of right/down paths."""
    rows, cols = len(grid), len(grid[0])
    dp = [0] * cols                                   # one row is enough
    dp[0] = 1 if grid[0][0] == 0 else 0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == 1:
                dp[c] = 0
            elif c > 0:
                dp[c] += dp[c - 1]                    # dp[c] still holds "from above"
    return dp[-1]

def paths_brute(grid, r=0, c=0):
    if r >= len(grid) or c >= len(grid[0]) or grid[r][c] == 1:
        return 0
    if (r, c) == (len(grid) - 1, len(grid[0]) - 1):
        return 1
    return paths_brute(grid, r + 1, c) + paths_brute(grid, r, c + 1)

assert unique_paths([[0] * 7 for _ in range(3)]) == 28 == math.comb(3 + 7 - 2, 3 - 1)
assert unique_paths([[0, 0, 0], [0, 1, 0], [0, 0, 0]]) == 2 and unique_paths([[1]]) == 0
for _ in range(200):
    rows, cols = rnd.randint(1, 6), rnd.randint(1, 6)
    g = [[1 if rnd.random() < 0.2 else 0 for _ in range(cols)] for _ in range(rows)]
    assert unique_paths(g) == paths_brute(g)
print("unique paths in a 10 x 10 empty grid:", unique_paths([[0] * 10 for _ in range(10)]), "=", math.comb(18, 9))

**Complexity:** O(m · n) time, O(n) space with one rolling row.

## Try it yourself

**1. Minimum path sum (LeetCode 64).** Grid of non-negative costs; move right or down; minimise the total. `[[1, 3, 1], [1, 5, 1], [4, 2, 1]]` → 7.

In [ ]:
# Solution — try it yourself first, then run this
def min_path_sum(grid):
    rows, cols = len(grid), len(grid[0])
    dp = [math.inf] * cols
    dp[0] = 0
    for r in range(rows):
        for c in range(cols):
            best_prev = dp[c] if c == 0 else min(dp[c], dp[c - 1])   # from above or from the left
            dp[c] = best_prev + grid[r][c]
    return dp[-1]

assert min_path_sum([[1, 3, 1], [1, 5, 1], [4, 2, 1]]) == 7
assert min_path_sum([[1, 2, 3], [4, 5, 6]]) == 12 and min_path_sum([[5]]) == 5
print(min_path_sum([[1, 3, 1], [1, 5, 1], [4, 2, 1]]))

**2. Partition equal subset sum (LeetCode 416).** Can the numbers be split into two groups with equal sums? Hint: it is a 0/1 knapsack with booleans — can some subset hit `total / 2`? Loop the sums downwards.

In [ ]:
# Solution — try it yourself first, then run this
def can_partition(nums):
    total = sum(nums)
    if total % 2:
        return False
    target = total // 2
    reachable = [True] + [False] * target            # reachable[s]: some subset sums to s
    for x in nums:
        for s in range(target, x - 1, -1):           # downwards: each number used once
            reachable[s] = reachable[s] or reachable[s - x]
    return reachable[target]

assert can_partition([1, 5, 11, 5]) and not can_partition([1, 2, 3, 5]) and can_partition([])
for _ in range(200):
    nums = [rnd.randint(1, 12) for _ in range(rnd.randint(0, 9))]
    brute = any(2 * sum(c) == sum(nums) for r in range(len(nums) + 1) for c in itertools.combinations(nums, r))
    assert can_partition(nums) == brute
print(can_partition([1, 5, 11, 5]))

**3. Word break (LeetCode 139).** Can the string be segmented into words from a vocabulary? `dp[i]` = "the first i characters can be segmented". Bonus: return one segmentation.

In [ ]:
# Solution — try it yourself first, then run this
def word_break(s, vocab):
    vocab, longest = set(vocab), max(map(len, vocab), default=0)
    dp = [True] + [False] * len(s)
    split_at = [None] * (len(s) + 1)
    for i in range(1, len(s) + 1):
        for j in range(max(0, i - longest), i):      # the last word is s[j:i]
            if dp[j] and s[j:i] in vocab:
                dp[i], split_at[i] = True, j
                break
    if not dp[-1]:
        return None
    words, i = [], len(s)
    while i > 0:
        words.append(s[split_at[i]:i]); i = split_at[i]
    return words[::-1]

print(word_break("unbelievablyfast", ["un", "believ", "ably", "able", "fast", "believe"]))
assert word_break("leetcode", ["leet", "code"]) == ["leet", "code"]
assert word_break("applepenapple", ["apple", "pen"]) == ["apple", "pen", "apple"]
assert word_break("catsandog", ["cats", "dog", "sand", "and", "cat"]) is None

## Say it in an interview

**30-second answer:** "I look for overlapping subproblems and optimal substructure: 'number of ways', 'min cost', 'longest subsequence', a take-or-skip choice at each step. Then I define the state in words, write the recurrence by thinking about the last decision, set the base cases, fill the table in dependency order and read off the answer. I usually sketch it top-down with memoisation, then convert to a bottom-up table and keep only the rows I need."

**Key facts:** coin change / knapsack are O(n · W), pseudo-polynomial · LIS is O(n²) or O(n log n) with patience sorting · LCS and edit distance are O(m · n) time and O(min(m, n)) space · unique paths = C(m + n − 2, m − 1).

**Follow-ups and traps:**
- *Loop order* in coin change decides combinations vs permutations; in 1-D knapsack, downwards = each item once, upwards = unlimited.
- *Greedy is not DP:* `[1, 3, 4]` coins, amount 6.
- *Top-down depth:* memoised recursion still recurses n deep in Python.
- *State too big?* Drop a dimension you can recompute, or compress (bitmask DP for n ≤ 20).

## Pattern cheat sheet

| Problem | State | Recurrence (last decision) | Time / space |
|---|---|---|---|
| climbing stairs | `dp[i]` ways to reach i | `dp[i-1] + dp[i-2]` | O(n) / O(1) |
| house robber | `dp[i]` best of first i | `max(dp[i-1], dp[i-2] + x)` | O(n) / O(1) |
| coin change (min) | `dp[a]` fewest coins | `1 + min(dp[a-c])` | O(A·C) / O(A) |
| coin change (ways) | `dp[a]` combinations | coins outer: `dp[a] += dp[a-c]` | O(A·C) / O(A) |
| 0/1 knapsack | `dp[w]` best value | downwards: `max(dp[w], dp[w-wt] + v)` | O(n·W) / O(W) |
| LIS | `tails[L]` smallest tail | binary search replace | O(n log n) / O(n) |
| LCS | `dp[i][j]` on prefixes | match: diag + 1, else max(up, left) | O(mn) / O(min) |
| edit distance | `dp[i][j]` edits on prefixes | match: diag, else 1 + min(3 neighbours) | O(mn) / O(min) |
| grid paths | `dp[r][c]` ways | up + left (0 on obstacles) | O(mn) / O(n) |
| word break | `dp[i]` prefix segmentable | any `dp[j]` and `s[j:i]` in vocab | O(n·L) / O(n) |

## Next

- [13 · Greedy and intervals](13_greedy_and_intervals.ipynb) — when a local choice is safe and DP is overkill.
- [08 · Recursion and backtracking](08_recursion_and_backtracking.ipynb) — memoisation is where DP starts.
- Theory: [DSA basics course — dynamic programming chapter](../../dsa_basics/index.html)